# Week 2: Embedding Pipeline & Classical Baseline (N=1000, auto-save)
**Project**: Quantum Kernels on CLIP / BERT Embeddings

**This version checkpoints after every expensive step** so a dead Colab session
never costs you more than the current step.

**Two layers of protection:**
1. **Google Drive mount** (recommended) — every save persists automatically
2. **Auto-download** after each step — browser download as backup

**Runtime**: ~12 minutes on a free Colab T4

In [ ]:
!pip install datasets transformers torch scikit-learn numpy --quiet

import numpy as np
import time, os, pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
import torch
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

print("All imports OK")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

---
## Step 0: Set up persistent storage + checkpoint helper

**Recommended:** mount Google Drive so saves survive a dead session automatically.
Set `USE_DRIVE = True`. Files go to `MyDrive/qml_project/`.

If you prefer not to use Drive, set `USE_DRIVE = False` and the notebook falls back
to local saves + browser auto-downloads.

In [ ]:
USE_DRIVE = True   # <-- set False to skip Drive and use downloads only

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

# Checkpoint helper: saves locally, copies to Drive (if on), and tries to download.
# The download is wrapped in try/except so a blocked browser popup never stops the run.
def checkpoint(filename, save_fn):
    """save_fn(path) writes the file to `path`. We then persist + try to download."""
    local_path = filename                      # always write a local copy first
    save_fn(local_path)

    # copy to Drive / save dir if different from local
    if SAVE_DIR not in ('.', '') and os.path.abspath(SAVE_DIR) != os.getcwd():
        dst = os.path.join(SAVE_DIR, os.path.basename(filename))
        import shutil
        shutil.copy(local_path, dst)
        print(f"  saved -> {dst}")
    else:
        print(f"  saved -> {local_path}")

    # try browser download (non-fatal if blocked)
    try:
        from google.colab import files
        files.download(local_path)
        print(f"  download triggered -> {os.path.basename(filename)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — file is still saved above)")

print("Checkpoint helper ready.")

---
## Step 1: Load SST-2 dataset

Extract 1100 train + 250 validation to build a real 1000 / 200 split.

In [ ]:
print("Loading SST-2 dataset...")
dataset = load_dataset("nyu-mll/glue", "sst2")   # namespaced path

N_TRAIN_EXTRACT = 1100
N_VAL_EXTRACT   = 250

train_data = dataset['train'].shuffle(seed=42).select(range(N_TRAIN_EXTRACT))
val_data   = dataset['validation'].shuffle(seed=42).select(range(N_VAL_EXTRACT))

print(f"Train: {len(train_data)} | Val: {len(val_data)}")
for i in range(3):
    ln = 'positive' if train_data[i]['label'] == 1 else 'negative'
    print(f"  [{ln}] {train_data[i]['sentence'][:70]}...")

---
## Step 2: Extract BERT embeddings — **checkpoint after this**

This is the most expensive step (~5 min). We save + download the raw embeddings
immediately so a session crash later never forces a re-extraction.

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
bert_model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
bert_model.eval()
print(f"{MODEL_NAME} loaded")

def extract_bert_embeddings(sentences, batch_size=32):
    out = []
    nb = (len(sentences) + batch_size - 1) // batch_size
    for i in range(0, len(sentences), batch_size):
        batch  = sentences[i:i+batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True,
                           max_length=128, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            o = bert_model(**inputs)
        out.append(o.last_hidden_state[:, 0, :].cpu().numpy())
        if (i // batch_size + 1) % 5 == 0:
            print(f"  batch {i//batch_size+1}/{nb}")
    return np.vstack(out)

print("Extracting train embeddings...")
t0 = time.time()
train_sentences = [x['sentence'] for x in train_data]
train_labels    = np.array([x['label'] for x in train_data])
train_emb = extract_bert_embeddings(train_sentences)
print(f"  done {time.time()-t0:.0f}s | {train_emb.shape}")

print("Extracting val embeddings...")
val_sentences = [x['sentence'] for x in val_data]
val_labels    = np.array([x['label'] for x in val_data])
val_emb = extract_bert_embeddings(val_sentences)
print(f"  done | {val_emb.shape}")

# ---- CHECKPOINT 1: raw BERT embeddings ----
print("\nCheckpoint 1 — raw embeddings:")
checkpoint('bert_raw_embeddings.npz',
           lambda p: np.savez(p, train_emb=train_emb, val_emb=val_emb,
                              train_labels=train_labels, val_labels=val_labels))

---
## Step 3: PCA + normalize — **checkpoint after this**

In [ ]:
N_QUBITS = 8
pca_dims_to_try = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims_to_try}

print("PCA variance explained:")
for n in pca_dims_to_try:
    print(f"  {n:>2} dims -> {np.sum(pca_objects[n].explained_variance_ratio_)*100:.1f}%")

pca_8       = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_val_pca   = pca_8.transform(val_emb)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_val_scaled   = np.clip(scaler.transform(X_val_pca), 0, np.pi)   # defensive clip

print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")
print(f"Val   range: [{X_val_scaled.min():.3f}, {X_val_scaled.max():.3f}]")

# ---- CHECKPOINT 2: PCA + scaler objects ----
print("\nCheckpoint 2 — PCA + scaler:")
checkpoint('pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler, 'n_qubits': N_QUBITS},
                                 open(p, 'wb')))

---
## Step 4: Classical baselines on real 1000 / 200 split — **checkpoint after this**

In [ ]:
N_TRAIN_SVM = 1000
N_TEST_SVM  = 200
assert len(X_train_scaled) >= N_TRAIN_SVM, f"Only {len(X_train_scaled)} train, need {N_TRAIN_SVM}"
assert len(X_val_scaled)   >= N_TEST_SVM,  f"Only {len(X_val_scaled)} val, need {N_TEST_SVM}"

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_val_scaled),   N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_val_scaled[idx_te],   val_labels[idx_te]

print(f"Train {len(X_tr)} | Test {len(X_te)} | pos rate train {np.mean(y_tr==1):.1%}")
print()

classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<12} {'Acc':>8} {'F1':>8}")
print("-"*30)
for name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    results[name] = {'accuracy': float(accuracy_score(y_te, y_pred)),
                     'f1': float(f1_score(y_te, y_pred))}
    print(f"{name:<12} {results[name]['accuracy']:>8.4f} {results[name]['f1']:>8.4f}")

# ---- CHECKPOINT 3: baseline results ----
print("\nCheckpoint 3 — baseline results:")
checkpoint('baseline_results.pkl',
           lambda p: pickle.dump(results, open(p, 'wb')))

---
## Step 5: PCA dimension sweep (no checkpoint — cheap to recompute)

In [ ]:
print(f"  {'Dims':>4}  {'Var':>7}  {'Acc':>8}  {'F1':>8}")
print("-"*32)
for n_dim in pca_dims_to_try:
    pca_d  = pca_objects[n_dim]
    Xtr_d  = pca_d.transform(train_emb[idx_tr])
    Xte_d  = pca_d.transform(val_emb[idx_te])
    sc_d   = MinMaxScaler(feature_range=(0, np.pi))
    Xtr_d  = sc_d.fit_transform(Xtr_d)
    Xte_d  = np.clip(sc_d.transform(Xte_d), 0, np.pi)
    clf_d  = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(Xtr_d, y_tr)
    yp     = clf_d.predict(Xte_d)
    var_d  = np.sum(pca_d.explained_variance_ratio_)*100
    print(f"  {n_dim:>4}  {var_d:>6.1f}%  {accuracy_score(y_te,yp):>8.4f}  {f1_score(y_te,yp):>8.4f}")

---
## Step 6: Final combined save — **checkpoint with shape verification**

This is the file Week 3 loads. Shape assertions guarantee a genuine N=1000 split.

In [ ]:
def save_final(p):
    np.savez(p,
             X_train=X_train_scaled, X_val=X_val_scaled,
             y_train=train_labels,   y_val=val_labels,
             X_tr_subset=X_tr, X_te_subset=X_te,
             y_tr_subset=y_tr, y_te_subset=y_te)

print("Checkpoint 4 — final combined embeddings:")
checkpoint('embeddings_week2.npz', save_final)

# verify
loaded = np.load('embeddings_week2.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8), "TRAIN SUBSET WRONG SIZE"
assert loaded['X_te_subset'].shape == (200, 8),  "TEST SUBSET WRONG SIZE"
print("\n  VERIFIED: genuine N=1000 / 200 split.")
print(f"\nAll files in {SAVE_DIR}:")
for f in os.listdir(SAVE_DIR):
    print(f"  {f}  ({os.path.getsize(os.path.join(SAVE_DIR, f))/1024:.0f} KB)")

---
## Week 2 complete

### Checkpoints saved (each downloaded + persisted to Drive):
1. `bert_raw_embeddings.npz` — after BERT extraction (the expensive step)
2. `pca_scaler.pkl` — after PCA + normalize
3. `baseline_results.pkl` — after classical baselines
4. `embeddings_week2.npz` — final combined file for Week 3

### If a session dies mid-run:
Re-mount Drive and the completed checkpoints are still in `MyDrive/qml_project/`.
You only redo the step that was running, not the whole notebook.

### Recovering from a checkpoint:
If you crash after BERT extraction, skip Step 2 next time and load instead:
```python
raw = np.load('/content/drive/MyDrive/qml_project/bert_raw_embeddings.npz')
train_emb, val_emb = raw['train_emb'], raw['val_emb']
train_labels, val_labels = raw['train_labels'], raw['val_labels']
```

### Week 3:
Genuine N=1000 kernel matrix takes ~83 min. If it returns in ~23 min,
you're still on 500 samples — check the shape assertion above passed.